In [18]:
import sys
sys.path.append('..')

import pandas as pd
from utils.db_utils import write_table, read_table

In [19]:
gdp_df = read_table("select * from sc_bronze.datagov_gdp")
gdp_df.head(20)

,state,date,sector,RM(million),growth_yoy
0,Johor,2016-01-01,Agriculture,15029.966,-3.718
1,Johor,2016-01-01,Mining and Quarrying,569.326,19.697
2,Johor,2016-01-01,Manufacturing,34121.545,5.448
3,Johor,2016-01-01,Construction,8978.486,23.515
4,Johor,2016-01-01,Services,56266.132,6.046
5,Johor,2017-01-01,Agriculture,16169.986,7.585
6,Johor,2017-01-01,Mining and Quarrying,655.514,15.139
7,Johor,2017-01-01,Manufacturing,36465.306,6.869
8,Johor,2017-01-01,Construction,8407.258,-6.362
9,Johor,2017-01-01,Services,59999.052,6.634


In [20]:
df_final = read_table("""
select
g.state as state,
g.date as date,
sum(g."RM(million)") as gdp_rm_million,
sum(g."RM(million)" * g.growth_yoy) / nullif(sum(g."RM(million)"), 0) as gdp_growth_yoy,
max(p.population) as population,
(sum(g."RM(million)") * 1000000.0 / nullif(max(p.population), 0)) as gdp_per_capita
from sc_bronze.datagov_gdp g
inner join sc_bronze.datagov_population p
on g.state = p.state
and g.date = p.date
group by g.state, g.date
order by g.state, g.date
""")

df_final

,state,date,gdp_rm_million,gdp_growth_yoy,population,gdp_per_capita
0,Johor,2016-01-01,114965.455,6.023904,3651800.0,31481.859631
1,Johor,2017-01-01,121697.116,5.978778,3697000.0,32917.802543
2,Johor,2018-01-01,128914.397,5.988315,3749400.0,34382.673761
3,Johor,2019-01-01,132617.111,3.589793,3761200.0,35259.255291
4,Johor,2020-01-01,126746.445,-3.751183,4009700.0,31609.957104
...,...,...,...,...,...,...
139,W.P. Putrajaya,2020-01-01,40697.536,-11.021000,109200.0,372688.058608
140,W.P. Putrajaya,2021-01-01,41895.612,2.944000,115200.0,363677.187500
141,W.P. Putrajaya,2022-01-01,42872.705,2.332000,117000.0,366433.376068
142,W.P. Putrajaya,2023-01-01,44503.210,3.803000,118800.0,374606.144781


In [21]:
write_table(df_final, 'sc_silver','gdp_per_capita')


Table sc_silver.gdp_per_capita written successfully.
